In [5]:
import pdb
import pickle
from itertools import product
from pathlib import Path
from functools import partial
import matplotlib.pyplot as plt

import jax
import jax.numpy as jnp
import numpy as np
import optimistix
from tqdm import tqdm

from cmfa.emu_example import clr_func, simple_net, ordered_ilr_func, closure, calculate_ci, gini_coefficient

jax.config.update("jax_enable_x64", True)


M = jnp.array([[1, -1, -2],[0, 0, -1],[-1, 0, 1]])
KNOWN_FLUXES = jnp.array([100.0])
TRUE_FREE_FLUXES = jnp.array([110.0, 20.0])
IX_INPUT_FLUX = np.array([0])
IX_FREE_FLUXES = np.array([1, 3])
IX_FIXED_FLUXES = np.array([2, 4, 5])

FREE_FLUX_RANGE = [[0.5*TRUE_FREE_FLUXES[0], 1.5*TRUE_FREE_FLUXES[0]], [0.5*TRUE_FREE_FLUXES[1], 1.5*TRUE_FREE_FLUXES[1]]]
FREE_FLUX_GRID_SIZE = [11, 11]

TOTAL_MOL = 1e5
SIGMA = 0.1
N_MEASUREMENT_PER_FLUX = 50
INITIAL_GUESS = jnp.array([100, 50])
SIMULATE_COMPOSITION = partial(simple_net, v1=KNOWN_FLUXES[0].item())
SEED = 1234


def get_flux_vector(free, known, ix_known, ix_free, M):
    n = known.shape[0] + free.shape[0] + M.shape[0]
    ix_known_or_free = jnp.concat([ix_free, ix_known])
    ix_fixed = jnp.delete(jnp.arange(n), ix_known_or_free)
    out = jnp.zeros(shape=(n,)).at[ix_known].set(known).at[ix_free].set(free)
    fixed = -M @ out[jnp.concat([ix_known, ix_free])]
    u = jnp.stack([-M, jnp.identity(len(ix_known_or_free))]) @ jnp.concat([known, free])
    return out.at[ix_fixed].set(fixed)

def simulate_measurement(key, yhat, sigma):
    err = jax.random.normal(key, shape=(yhat.shape)) * sigma
    log_mst = jnp.log(yhat) + err
    return jnp.exp(log_mst)


simulate_measurements = jax.vmap(simulate_measurement, in_axes=(0, None, None))


def wssr(y, args):
    target, weights = args
    return jnp.sum(weights * (y - target) ** 2)


def loss_trad(log_free_flux, args):
    composition = SIMULATE_COMPOSITION(jnp.exp(log_free_flux))
    return wssr(composition, args), composition


def loss_clr(log_free_flux, args):
    composition = SIMULATE_COMPOSITION(jnp.exp(log_free_flux))
    target, weights = args
    return wssr(clr_func(composition), (clr_func(target), weights)), composition


def loss_ilr(log_free_flux, args):
    composition = SIMULATE_COMPOSITION(jnp.exp(log_free_flux))
    target, weights = args
    return wssr(
        ordered_ilr_func(composition), (ordered_ilr_func(target), weights)
    ), composition


In [3]:

flux_from_free = partial(
    get_flux_vector,
    known=KNOWN_FLUXES,
    ix_known=IX_INPUT_FLUX,
    ix_free=IX_FREE_FLUXES,
    M=M,
)
solver = optimistix.BFGS(rtol=1e-8, atol=1e-8)
true_flux = flux_from_free(TRUE_FREE_FLUXES)
true_id = SIMULATE_COMPOSITION(TRUE_FREE_FLUXES)
n_flux = len(true_flux)
n_comp = len(true_id)

flux_ranges = [
    np.linspace(range_i[0], range_i[1], grid_size_i)
    for range_i, grid_size_i in zip(FREE_FLUX_RANGE, FREE_FLUX_GRID_SIZE)
]
free_fluxes = jnp.array(list(product(*flux_ranges)))

key = jax.random.key(seed=SEED)

true_ids = jnp.empty((free_fluxes.shape[0], n_comp))
for i, free_flux in enumerate(tqdm(free_fluxes)):
    key, _ = jax.random.split(key)
    keys = jax.random.split(key, N_MEASUREMENT_PER_FLUX) 
    true_ids = true_ids.at[i].set(SIMULATE_COMPOSITION(free_flux))

100%|██████████| 121/121 [00:01<00:00, 60.84it/s]


In [8]:
true_ids.shape

(121, 4)

In [12]:
indices = np.where(true_ids < 0)

In [13]:
indices

(array([ 0,  1,  2,  3,  4,  5,  6, 11, 12, 13, 14, 22]),
 array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]))

In [16]:
true_ids[1,:]

Array([-9.89123918e-06,  8.47270936e-01,  1.52485865e-01,  2.53090120e-04],      dtype=float64)

In [ ]:
print(f"Fluxes producing negative isotopologue [v2,v4]: \nfrom {free_fluxes[0]} to {free_fluxes[6]}, \nfrom {free_fluxes[11]} to {free_fluxes[14]}, \n{free_fluxes[22]}")


fluxes producing negative isotopologue: 
from [55. 10.] to [55. 22.], 
from [66. 10.] to [66. 16.], 
[77. 10.]
